Load clinical data, we only care about the last follow up of the patient, and the survival status of the last follow up:

In [1]:
import pandas as pd

clinical_data_df = pd.read_csv("/data1/r20user3/shared_project/Hist2Cell/data/brca_clinical_from_cell_paper/brca_clinical_from_tcga.tsv", sep="\t")
clinical_data_df.head(2)

,case_id,case_submitter_id,project_id,age_at_index,age_is_obfuscated,cause_of_death,cause_of_death_source,country_of_residence_at_enrollment,days_to_birth,days_to_death,...,treatment_arm,treatment_dose,treatment_dose_units,treatment_effect,treatment_effect_indicator,treatment_frequency,treatment_intent_type,treatment_or_therapy,treatment_outcome,treatment_type
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,TCGA-BRCA,60,'--,'--,'--,'--,-22279,'--,...,'--,'--,'--,'--,'--,'--,'--,no,'--,"Radiation Therapy, NOS"
1,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,TCGA-BRCA,60,'--,'--,'--,'--,-22279,'--,...,'--,'--,'--,'--,'--,'--,'--,yes,'--,"Pharmaceutical Therapy, NOS"


In [2]:
selected_clinical_data_df = clinical_data_df[[
    'case_submitter_id',
    'days_to_death',
    'days_to_last_follow_up',
    'vital_status',
]]
selected_clinical_data_df = selected_clinical_data_df.drop_duplicates()
selected_clinical_data_df

,case_submitter_id,days_to_death,days_to_last_follow_up,vital_status
0,TCGA-E2-A1IU,'--,337,Alive
2,TCGA-A1-A0SB,'--,259,Alive
4,TCGA-A2-A04W,'--,3102,Alive
6,TCGA-AN-A0AM,'--,5,Alive
8,TCGA-LL-A440,'--,759,Alive
...,...,...,...,...
2185,TCGA-A2-A0CP,'--,2813,Alive
2187,TCGA-PL-A8LX,'--,5,Alive
2189,TCGA-A2-A3XZ,'--,1532,Alive
2191,TCGA-E9-A295,'--,375,Alive


We decide the `censor` label for each patient, and create `survival days` column, for patient with no record, we set `pandas.NA`.

censored: 
- 0 - already dead at last report; 
- 1 - still alive at last report;

In [3]:
# the following code is not perfecly written, but it works.
# if you feel it is hard to understand, just skip to the results.
# you can also improve the code if you want :)

import pandas
import json

selected_clinical_data_df = selected_clinical_data_df.rename(columns={'case_submitter_id': 'submitter_id'})
selected_clinical_data_df = selected_clinical_data_df.drop_duplicates(keep="first")
selected_clinical_data_df = selected_clinical_data_df.replace('\'--', pandas.NA)
selected_clinical_data_df = selected_clinical_data_df.to_json(orient="index")
selected_clinical_data_df = json.loads(selected_clinical_data_df)

for sample in selected_clinical_data_df:
    if selected_clinical_data_df[sample]['vital_status'] == 'Alive':
        try:
            survival_days = int(selected_clinical_data_df[sample]['days_to_last_follow_up'])
            censor = 1
        except:
            survival_days = pandas.NA
            censor = pandas.NA
    elif selected_clinical_data_df[sample]['vital_status'] == 'Dead':
        try:
            survival_days = int(selected_clinical_data_df[sample]['days_to_death'])
            censor = 0
        except:
            survival_days = pandas.NA
            censor = pandas.NA
    else:
        survival_days = pandas.NA
        censor = pandas.NA
    selected_clinical_data_df[sample]['survival_days'] = survival_days
    selected_clinical_data_df[sample]['censor'] = censor
    
selected_clinical_data_df = pd.DataFrame.from_dict(selected_clinical_data_df, orient='index')
selected_clinical_data_df

,submitter_id,days_to_death,days_to_last_follow_up,vital_status,survival_days,censor
0,TCGA-E2-A1IU,None,337,Alive,337,1
2,TCGA-A1-A0SB,None,259,Alive,259,1
4,TCGA-A2-A04W,None,3102,Alive,3102,1
6,TCGA-AN-A0AM,None,5,Alive,5,1
8,TCGA-LL-A440,None,759,Alive,759,1
...,...,...,...,...,...,...
2185,TCGA-A2-A0CP,None,2813,Alive,2813,1
2187,TCGA-PL-A8LX,None,5,Alive,5,1
2189,TCGA-A2-A3XZ,None,1532,Alive,1532,1
2191,TCGA-E9-A295,None,375,Alive,375,1


Now we create `Y - discrete label of survival length`, I use 4 intervals here following `MCAT.pdf` paper. (I saved the paper in this same folder)

In [4]:
import numpy as np


selected_clinical_data_df.dropna(subset=['survival_days'], inplace=True)
survival_days_list = np.array(list(selected_clinical_data_df['survival_days']))
quantile_25 = np.percentile(survival_days_list, 25)
quantile_50 = np.percentile(survival_days_list, 50)
quantile_75 = np.percentile(survival_days_list, 75)

In [5]:
selected_clinical_data_df['survival_interval'] = np.ones(selected_clinical_data_df.shape[0], dtype=np.int)

for i in range(0, len(selected_clinical_data_df)):
    survival_time = selected_clinical_data_df.iloc[i]['survival_days']

    if survival_time < quantile_25:
        selected_clinical_data_df.iloc[i, -1] = 0
    elif quantile_25 <= survival_time and survival_time < quantile_50:
        selected_clinical_data_df.iloc[i, -1] = 1
    elif quantile_50 <= survival_time and survival_time < quantile_75:
        selected_clinical_data_df.iloc[i, -1] = 2
    elif quantile_75 <= survival_time:
        selected_clinical_data_df.iloc[i, -1] = 3
    else:
        selected_clinical_data_df.iloc[i, -1] = pandas.NA
        
selected_clinical_data_df

/tmp/ipykernel_2811726/1240998653.py:1: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  selected_clinical_data_df['survival_interval'] = np.ones(selected_clinical_data_df.shape[0], dtype=np.int)


,submitter_id,days_to_death,days_to_last_follow_up,vital_status,survival_days,censor,survival_interval
0,TCGA-E2-A1IU,None,337,Alive,337,1,0
2,TCGA-A1-A0SB,None,259,Alive,259,1,0
4,TCGA-A2-A04W,None,3102,Alive,3102,1,3
6,TCGA-AN-A0AM,None,5,Alive,5,1,0
8,TCGA-LL-A440,None,759,Alive,759,1,1
...,...,...,...,...,...,...,...
2185,TCGA-A2-A0CP,None,2813,Alive,2813,1,3
2187,TCGA-PL-A8LX,None,5,Alive,5,1,0
2189,TCGA-A2-A3XZ,None,1532,Alive,1532,1,2
2191,TCGA-E9-A295,None,375,Alive,375,1,0


Now, we add the subtype label from `mmc2.xlsx` to the above `selected_clinical_data_df`

In [6]:
cell_paper_df = pd.read_excel("/data1/r20user3/shared_project/Hist2Cell/data/brca_clinical_from_cell_paper/mmc2.xlsx", engine='openpyxl', skiprows=2)
cell_paper_df[['Case.ID', 'PAM50']]
cell_paper_df.head(2)

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Case.ID,Final Pathology,PAM50,TumorPurity,ProliferationScore,mRNA,miRNA,SNP,Exome,Methylation,...,Hormone_a score,Hormone_b score,PI3K/Akt score,Ras/MAPK score,RTK score,TSC/mTOR score,ISOpure,OncoSign,ElasticNet,Unnamed: 27
0,TCGA-BH-A18G,IDC,Basal,0.79,0.423067,TCGA-BH-A18G-01A-11R-A12D-07,TCGA-BH-A18G-01A-11R-A12C-13,TCGA-BH-A18G-01A-11D-A12A-01,TCGA-BH-A18G-01A-11D-A12B-09,TCGA-BH-A18G-01A-11D-A12E-05,...,-1.873243,-0.120416,-3.469006,0.94358,-1.421711,1.376493,NaN,NaN,NaN,IDC-like
1,TCGA-A1-A0SP,IDC,Basal,0.46,0.262933,TCGA-A1-A0SP-01A-11R-A084-07,TCGA-A1-A0SP-01A-11R-A085-13,TCGA-A1-A0SP-01A-11D-A087-01,TCGA-A1-A0SP-01A-11D-A099-09,TCGA-A1-A0SP-01A-11D-A10P-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ILC-like


In [7]:
cell_paper_df['PAM50'].value_counts()

PAM50
LumA      415
LumB      176
Basal     136
Her2       65
Normal     25
Name: count, dtype: int64

In [8]:
selected_clinical_data_df = selected_clinical_data_df.rename(columns={'submitter_id': 'Case.ID'})
selected_clinical_data_df

,Case.ID,days_to_death,days_to_last_follow_up,vital_status,survival_days,censor,survival_interval
0,TCGA-E2-A1IU,None,337,Alive,337,1,0
2,TCGA-A1-A0SB,None,259,Alive,259,1,0
4,TCGA-A2-A04W,None,3102,Alive,3102,1,3
6,TCGA-AN-A0AM,None,5,Alive,5,1,0
8,TCGA-LL-A440,None,759,Alive,759,1,1
...,...,...,...,...,...,...,...
2185,TCGA-A2-A0CP,None,2813,Alive,2813,1,3
2187,TCGA-PL-A8LX,None,5,Alive,5,1,0
2189,TCGA-A2-A3XZ,None,1532,Alive,1532,1,2
2191,TCGA-E9-A295,None,375,Alive,375,1,0


In [9]:
merged_df = pd.merge(selected_clinical_data_df, cell_paper_df[['Case.ID', 'PAM50']], on='Case.ID', how='left')
merged_df.head(2)

,Case.ID,days_to_death,days_to_last_follow_up,vital_status,survival_days,censor,survival_interval,PAM50
0,TCGA-E2-A1IU,None,337,Alive,337,1,0,LumA
1,TCGA-A1-A0SB,None,259,Alive,259,1,0,Normal


In [10]:
merged_df

,Case.ID,days_to_death,days_to_last_follow_up,vital_status,survival_days,censor,survival_interval,PAM50
0,TCGA-E2-A1IU,None,337,Alive,337,1,0,LumA
1,TCGA-A1-A0SB,None,259,Alive,259,1,0,Normal
2,TCGA-A2-A04W,None,3102,Alive,3102,1,3,Her2
3,TCGA-AN-A0AM,None,5,Alive,5,1,0,LumB
4,TCGA-LL-A440,None,759,Alive,759,1,1,LumA
...,...,...,...,...,...,...,...,...
1091,TCGA-A2-A0CP,None,2813,Alive,2813,1,3,LumA
1092,TCGA-PL-A8LX,None,5,Alive,5,1,0,NaN
1093,TCGA-A2-A3XZ,None,1532,Alive,1532,1,2,Her2
1094,TCGA-E9-A295,None,375,Alive,375,1,0,LumA


We save the survival labels into 3 different subtyps:
- `HER2+`: `Her2`;
- `Luminal`: `LumA` and `LumB`;
- `Basal`: `Basal`;

In [11]:
# we filtered out the cases that are not in the cell paper

from glob import glob

brca_filtered_case_path_list = glob("/data1/r20user3/shared_project/Hist2Cell/data/brca/*")
brca_filtered_case_id_list = [case_path.split("/")[-1] for case_path in brca_filtered_case_path_list]
brca_filtered_case_id_df = pd.DataFrame(brca_filtered_case_id_list, columns=['Slide.ID'])
brca_filtered_case_id_df['Case.ID'] = brca_filtered_case_id_df['Slide.ID'].apply(lambda x: x.split('-01Z')[0])
merged_df = pd.merge(brca_filtered_case_id_df, merged_df, on='Case.ID', how='left')
len(merged_df)

826

In [12]:
merged_df['PAM50'].value_counts()

PAM50
LumA      313
LumB      114
Basal     111
Her2       54
Normal     21
Name: count, dtype: int64

In [ ]:
merged_df.to_csv("brca_survival_label_merge.csv")

In [52]:
selected_df = merged_df[(merged_df['PAM50'] == 'LumA') | (merged_df['PAM50'] == 'LumB')]
selected_df.to_csv("brca_survival_label_Luminal.csv")

In [49]:
selected_df = merged_df[merged_df['PAM50'] == 'Basal']
selected_df.to_csv("brca_survival_label_Basal.csv")

In [50]:
selected_df = merged_df[merged_df['PAM50'] == 'Her2']
selected_df.to_csv("brca_survival_label_HER2.csv")